# Download and load the Qwen2.5-0.5B-Instruct model and tokenizer

In [81]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Hugging Face model identifier
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Download and load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Download and load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print(model.device)  # Check the device of the model
print(model.training)  # Check if the model is in training mode

print("Model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

cuda:0
False
Model and tokenizer loaded successfully!


# Create the Chatbot's Introduction

In [82]:
bob_introduction = "System: Bob is an amazing chatbot. It knows everything about movies and its incredibly helpful."

# Create the generate method

In [83]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Define the Chatbot / Implement RAG

In [84]:
import pandas as pd
from sentence_transformers import SentenceTransformer

class BobTheChatbot:
    def __init__(self, model, tokenizer, introduction=bob_introduction, max_answer_length=200):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length

        # RAG setup
        df = pd.read_csv('wiki_movie_plots_deduped.csv', usecols=['Release Year', 'Title', 'Plot'])
        self.docs = (df["Title"] + " (" + df["Release Year"].astype(str) + "): " + df["Plot"]).tolist()
        self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.sentence_model.encode(self.docs, convert_to_tensor=True)

    def retrieve_relevant_movie_information(self, prompt):
        prompt_embedding = self.sentence_model.encode(prompt, convert_to_tensor=True)
        cosine_scores = torch.nn.functional.cosine_similarity(prompt_embedding, self.embeddings)
        top_k_indices = torch.topk(cosine_scores, k=1).indices
        relevant_movie_information = self.docs[top_k_indices[0]]
        return relevant_movie_information

    def chatWithRAG(self, prompt):
        relevant_movie_information = self.retrieve_relevant_movie_information(prompt)
        self.context += "\nSystem: Use the following movie information\n" + relevant_movie_information + "\nMe: " + prompt + "\nAnswer in 3-4 sentences\nBob: "
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=300)
            answer = extended[start_index:]
            if ("\nMe: " in answer or "\nSystem: " in answer or extended == context or len(answer) >= self.max_answer_length): break
            context = extended
        answer = answer.split("\nMe: ")[0]
        answer = answer.split("\nSystem: ")[0]
        answer = answer.split("Answer in 3-4 sentences")[0] # Hack to ensure the answer is concise and doesn't include the instruction
        self.context += answer
        return answer.strip()

# Create the Chatbot

In [85]:
bob = BobTheChatbot(model, tokenizer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Call Chat

In [86]:
bob.chatWithRAG("Hello Bob! Can you tell me something about the movie Handphone?")

"1. The movie is about a talented actor named Seung-min, who has lost hope of turning his life around due to a sex tape that he obtained from a former lover.\n2. Seung-min discovers that his wife has been tracking down the culprit behind the sex tape, and now she is demanding that he return his phone to retrieve the evidence.\n3. Despite his efforts to recover the phone, Seung-min finds himself unable to escape the demands of Yi-gyu, the woman who found his phone. \n4. As a result, Seung-min must navigate through a series of challenges and obstacles to finally locate the missing phone and bring it back to him. \n\nWhat does this mean? This means that even after all the twists and turns, Bob remains determined to help Seung-min find the missing phone and restore his life. Although he may face numerous challenges along the way, he will keep trying until the end. Ultimately, Bob's love for Seung-min will triumph over any obstacle that comes his way. The movie hints at themes such as perse

In [87]:
bob.chatWithRAG("Can you tell me about the movie The Dark Knight?")

"1. The movie is set in the world of medieval Europe during the Hundred Years' War between England and France.\n2. The main character, the Dark Knight, is a knight who returns home after being captured by the French forces and is sent to prison.\n3. The Dark Knight is a skilled swordsman and uses his knowledge of the war to defend himself and his fellow prisoners.\n4. After a series of events, including a battle with a giant dragon, the Dark Knight gains the trust of the prison guards and is eventually released.\n5. The Dark Knight's journey to freedom is interrupted when he discovers that his father has died in the war and that his mother is planning to marry the king of France.\n6. The Dark Knight must use his skills and knowledge to try to save his family and ultimately free himself from imprisonment.\n\nWhat does this mean? This means that the Dark Knight's journey to freedom is fraught with danger and uncertainty. Despite his bravery and intelligence, he must confront difficult ch